<a href="https://colab.research.google.com/github/ericyoc/agentic_ai_scrypt_cracking_poc/blob/main/agentic_ai_scrypt_cracking_poc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Agent scrypt Cracking in Google Colab

This notebook is the scrypt version of the multi-agent proof of concept.

## Architecture
- **1 Coordinator Agent**
- **4 Worker Agents**
- **1 Verifier Agent**

Only the **first user** in the supplied `shadow_scrypt.pdf` is attacked.

The attached shadow format is:

`User:$scrypt$N$r$p$Salt$Base64Hash`

The NLTK candidate list is built once, divided into four non-overlapping shards, and the four MCP workers test those shards concurrently with Python's `hashlib.scrypt()`.

In [ ]:
# IMPORTANT:
# Run this first in a fresh Colab runtime.

!pip -q install --upgrade \
    "mcp<2" \
    "nltk==3.10.2" \
    pypdf \
    transformers \
    accelerate \
    huggingface_hub \
    pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 121.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 51.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatib

In [ ]:
# Verify dependency versions immediately.

import importlib.metadata

mcp_version = importlib.metadata.version("mcp")
nltk_version = importlib.metadata.version("nltk")

print("MCP version :", mcp_version)
print("NLTK version:", nltk_version)

assert int(mcp_version.split(".")[0]) == 1, (
    f"Expected MCP 1.x, found {mcp_version}"
)

assert nltk_version == "3.10.2", (
    f"Expected NLTK 3.10.2, found {nltk_version}"
)

print("Dependency versions verified.")

MCP version : 1.30.0
NLTK version: 3.10.2
Dependency versions verified.


In [ ]:
# Mount Google Drive and create project folders.

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import asyncio
import base64
import hashlib
import json
import os
import re
import shutil
import sys
import time

PROJECT = Path(
    "/content/drive/MyDrive/Task2_Efficient_MultiAgent_Scrypt"
)

INPUTS = PROJECT / "inputs"
SRC = PROJECT / "src"
MODELS = PROJECT / "models"
RESULTS = PROJECT / "results"
NLTK_DATA_DIR = PROJECT / "nltk_data"
PROGRESS = PROJECT / "progress"

for folder in (
    INPUTS,
    SRC,
    MODELS,
    RESULTS,
    NLTK_DATA_DIR,
    PROGRESS,
):
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

NUM_WORKER_AGENTS = 4

print("Project:", PROJECT)
print("Worker agents:", NUM_WORKER_AGENTS)

Mounted at /content/drive
Project: /content/drive/MyDrive/Task2_Efficient_MultiAgent_Scrypt
Worker agents: 4


## Download and verify the NLTK `words` corpus

This runs **before** the cracking workflow.

The corpus is stored in Google Drive so later Colab runs can reuse it.

In [ ]:
import nltk

# Put our persistent Drive-backed NLTK directory first.
if str(NLTK_DATA_DIR) not in nltk.data.path:
    nltk.data.path.insert(
        0,
        str(NLTK_DATA_DIR),
    )

print("NLTK search paths:")
for p in nltk.data.path:
    print(" -", p)

# Download only if the corpus is not already present.
try:
    nltk.data.find("corpora/words")
    print("NLTK words corpus already present.")

except LookupError:
    print("Downloading NLTK words corpus...")

    ok = nltk.download(
        "words",
        download_dir=str(NLTK_DATA_DIR),
        quiet=False,
    )

    if not ok:
        raise RuntimeError(
            "NLTK downloader reported failure."
        )

# Hard verification.
nltk.data.find("corpora/words")

from nltk.corpus import words

test_words = words.words()

if not test_words:
    raise RuntimeError(
        "NLTK words corpus loaded but is empty."
    )

print(
    "NLTK words corpus verified:",
    len(test_words),
    "entries"
)

NLTK search paths:
 - /content/drive/MyDrive/Task2_Efficient_MultiAgent_Scrypt/nltk_data
 - /root/nltk_data
 - /usr/nltk_data
 - /usr/share/nltk_data
 - /usr/lib/nltk_data
 - /usr/share/nltk_data
 - /usr/local/share/nltk_data
 - /usr/lib/nltk_data
 - /usr/local/lib/nltk_data


[nltk_data] Downloading package words to /content/drive/MyDrive/Task2_
[nltk_data]     Efficient_MultiAgent_Scrypt/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.


NLTK words corpus verified: 236736 entries


## Upload/use the supplied shadow PDF

If the PDF already exists in Drive at:

`MyDrive/Task2_Efficient_MultiAgent_Scrypt/inputs/shadow_scrypt.pdf`

it is reused.

In [ ]:
from google.colab import files

shadow_path = INPUTS / "shadow_scrypt.pdf"

if not shadow_path.exists():

    print("Upload shadow_scrypt.pdf")

    uploaded = files.upload()

    pdf_name = next(
        (
            name
            for name in uploaded.keys()
            if name.lower().endswith(".pdf")
        ),
        None,
    )

    if pdf_name is None:
        raise FileNotFoundError(
            "No PDF was uploaded."
        )

    shutil.copy2(
        pdf_name,
        shadow_path,
    )

print("Using shadow file:", shadow_path)

Upload shadow_scrypt.pdf


Saving shadow_scrypt.pdf to shadow_scrypt.pdf
Using shadow file: /content/drive/MyDrive/Task2_Efficient_MultiAgent_Scrypt/inputs/shadow_scrypt.pdf


In [ ]:
# Parse every scrypt user from the supplied PDF.

from pypdf import PdfReader
import pandas as pd


def parse_shadow_pdf(pdf_path):

    reader = PdfReader(
        str(pdf_path)
    )

    text = "\n".join(
        (page.extract_text() or "")
        for page in reader.pages
    )

    entries = []

    for raw in text.splitlines():

        line = raw.strip().strip('"')

        if ":$scrypt$" not in line:
            continue

        user, scrypt_record = line.split(
            ":",
            1,
        )

        parts = scrypt_record.split("$")

        # Expected:
        # ["", "scrypt", N, r, p, salt, base64_hash]
        if len(parts) != 7:
            continue

        _, algorithm, n_text, r_text, p_text, salt, hash_b64 = parts

        if algorithm != "scrypt":
            continue

        entries.append(
            {
                "User": user,
                "Algorithm": algorithm,
                "N": int(n_text),
                "r": int(r_text),
                "p": int(p_text),
                "Salt": salt,
                "Hash value": hash_b64,
                "scrypt_record": scrypt_record,
            }
        )

    return entries


shadow_users = parse_shadow_pdf(
    shadow_path
)

if not shadow_users:
    raise RuntimeError(
        "No scrypt entries were parsed."
    )

df = pd.DataFrame(
    shadow_users
)

display(
    df[
        [
            "User",
            "Algorithm",
            "N",
            "r",
            "p",
            "Salt",
            "Hash value",
        ]
    ]
)

first = shadow_users[0]

print()
print(
    "Target first user:",
    first["User"],
)

print(
    "scrypt parameters:",
    f"N={first['N']}, r={first['r']}, p={first['p']}",
)

,User,Algorithm,N,r,p,Salt,Hash value
0,Bilbo,scrypt,16384,8,1,J9FW66ZdPI2nrIMcOxFYI.,zYNn52jfupTkrF/puqvbe7h7cFx3TelLEy74jZESjOY=
1,Gandalf,scrypt,16384,8,1,J9FW66ZdPI2nrIMcOxFYI.,Tw/52I9jogqq4AqE9OOjHLRS92SKunOfe4oO89dkOIc=
2,Thorin,scrypt,16384,8,1,J9FW66ZdPI2nrIMcOxFYI.,lkuTTVZL11Pl+IwY+gMO/jLpoVWOKnOEfiwyqLq+bc0=
3,Fili,scrypt,32768,8,1,M9xNRFBDn0pUkPKIVCSBzu,EXfqrafcdzSXa28S6S0rRXGuHkbaRc+ySyv1smrZIV4=
4,Kili,scrypt,32768,8,1,M9xNRFBDn0pUkPKIVCSBzu,oryff5RM3eBUL4aMBB5Q0zYpVxMqTm0t7JB28RAc20g=
5,Balin,scrypt,65536,8,1,xGKjb94iwmlth954hEaw3O,HXNJKACWTtPCjfKFqlG8IXAH+yAtSrJ/uIDlw7AM2EY=
6,Dwalin,scrypt,65536,8,1,xGKjb94iwmlth954hEaw3O,J1Rxpct1kEiIwHjqYdd8MJuxke3fFrtUV6M+5LciHlw=
7,Oin,scrypt,65536,8,1,xGKjb94iwmlth954hEaw3O,GDYdMKcqQ4bWqlX25vcvtlGUeqkemV8ZeGCJGaEG4M4=
8,Gloin,scrypt,131072,8,1,/8UByex2ktrWATZOBLZ0Du,EVmDgM8sRa0VhTJ9A6is4i8TEjt2j1T/yzwKoK811nc=
9,Dori,scrypt,131072,8,1,/8UByex2ktrWATZOBLZ0Du,UixXzi2vrI/lYq1qX/jotZmwX+Ysi+xZQN5oz7RdcuA=



Target first user: Bilbo
scrypt parameters: N=16384, r=8, p=1


## Build the candidate list ONCE

The assignment constrains passwords to one NLTK word between 6 and 10 letters.

In [ ]:
from nltk.corpus import words

corpus = words.words()

candidates = [
    word
    for word in corpus
    if 6 <= len(word) <= 10
    and word.isalpha()
]

if not candidates:
    raise RuntimeError(
        "Candidate list is empty."
    )

candidate_path = (
    INPUTS
    / "candidates_6_10.txt"
)

candidate_path.write_text(
    "\n".join(candidates)
)

print(
    "Candidate count:",
    len(candidates),
)

print(
    "Candidate file:",
    candidate_path,
)

shards = [
    candidates[i::NUM_WORKER_AGENTS]
    for i in range(NUM_WORKER_AGENTS)
]

shard_files = []

for i, shard in enumerate(shards):

    shard_path = (
        INPUTS
        / f"candidate_shard_{i}.txt"
    )

    shard_path.write_text(
        "\n".join(shard)
    )

    shard_files.append(
        shard_path
    )

    print(
        f"Worker {i+1} shard:",
        len(shard),
        "candidates",
    )

Candidate count: 135145
Candidate file: /content/drive/MyDrive/Task2_Efficient_MultiAgent_Scrypt/inputs/candidates_6_10.txt
Worker 1 shard: 33787 candidates
Worker 2 shard: 33786 candidates
Worker 3 shard: 33786 candidates
Worker 4 shard: 33786 candidates


## Cache the local Hugging Face agent model

Model: `Qwen/Qwen2.5-0.5B-Instruct`

In [ ]:
from huggingface_hub import snapshot_download

MODEL_ID = (
    "Qwen/Qwen2.5-0.5B-Instruct"
)

MODEL_DIR = (
    MODELS
    / "Qwen2.5-0.5B-Instruct"
)

if not (
    MODEL_DIR
    / "config.json"
).exists():

    print(
        "Downloading model to Google Drive..."
    )

    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=str(MODEL_DIR),
    )

else:

    print(
        "Model already cached."
    )

print(
    "Model directory:",
    MODEL_DIR,
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Model directory: /content/drive/MyDrive/Task2_Efficient_MultiAgent_Scrypt/models/Qwen2.5-0.5B-Instruct


## Create `mcp_server.py`

Each worker receives one prebuilt shard file. The MCP worker does **not** download NLTK or rebuild the corpus.

In [ ]:
mcp_code = '\nfrom pathlib import Path\nimport base64\nimport hashlib\nimport json\nimport time\n\nfrom pypdf import PdfReader\nfrom mcp.server.fastmcp import FastMCP\n\n\nPROJECT = Path(\n    "/content/drive/MyDrive/Task2_Efficient_MultiAgent_Scrypt"\n)\n\nSHADOW_PDF = (\n    PROJECT\n    / "inputs"\n    / "shadow_scrypt.pdf"\n)\n\nRESULTS = PROJECT / "results"\nPROGRESS = PROJECT / "progress"\n\nRESULTS.mkdir(\n    parents=True,\n    exist_ok=True,\n)\n\nPROGRESS.mkdir(\n    parents=True,\n    exist_ok=True,\n)\n\nmcp = FastMCP(\n    "efficient-multiagent-scrypt"\n)\n\n\ndef _parse_shadow_pdf():\n\n    reader = PdfReader(\n        str(SHADOW_PDF)\n    )\n\n    text = "\\n".join(\n        (page.extract_text() or "")\n        for page in reader.pages\n    )\n\n    entries = []\n\n    for raw in text.splitlines():\n\n        line = raw.strip().strip(\'"\')\n\n        if ":$scrypt$" not in line:\n            continue\n\n        user, scrypt_record = line.split(\n            ":",\n            1,\n        )\n\n        parts = scrypt_record.split("$")\n\n        if len(parts) != 7:\n            continue\n\n        _, algorithm, n_text, r_text, p_text, salt, hash_b64 = parts\n\n        if algorithm != "scrypt":\n            continue\n\n        entries.append(\n            {\n                "user": user,\n                "algorithm": algorithm,\n                "n": int(n_text),\n                "r": int(r_text),\n                "p": int(p_text),\n                "salt": salt,\n                "hash_b64": hash_b64,\n            }\n        )\n\n    return entries\n\n\ndef _verify_candidate(\n    password,\n    target,\n):\n\n    expected = base64.b64decode(\n        target["hash_b64"]\n    )\n\n    derived = hashlib.scrypt(\n        password.encode("utf-8"),\n        salt=target["salt"].encode("utf-8"),\n        n=target["n"],\n        r=target["r"],\n        p=target["p"],\n        dklen=len(expected),\n        maxmem=1024 * 1024 * 1024,\n    )\n\n    return derived == expected\n\n\ndef _write_progress(\n    agent_id,\n    attempts,\n    shard_size,\n    start_time,\n    status,\n    found=False,\n    password=None,\n):\n\n    elapsed = max(\n        time.perf_counter() - start_time,\n        1e-9,\n    )\n\n    rate = attempts / elapsed\n\n    remaining = max(\n        shard_size - attempts,\n        0,\n    )\n\n    eta_seconds = (\n        remaining / rate\n        if rate > 0\n        else None\n    )\n\n    progress = {\n        "agent_id": agent_id,\n        "status": status,\n        "attempts": attempts,\n        "shard_size": shard_size,\n        "percent_complete": (\n            100.0 * attempts / shard_size\n            if shard_size\n            else 100.0\n        ),\n        "elapsed_seconds": elapsed,\n        "checks_per_second": rate,\n        "eta_seconds": eta_seconds,\n        "found": found,\n        "password": password,\n    }\n\n    tmp = PROGRESS / f"{agent_id}.tmp"\n    final = PROGRESS / f"{agent_id}.json"\n\n    tmp.write_text(\n        json.dumps(\n            progress,\n            indent=2,\n        )\n    )\n\n    tmp.replace(\n        final\n    )\n\n\n@mcp.tool()\ndef get_first_user() -> dict:\n\n    entries = _parse_shadow_pdf()\n\n    if not entries:\n        raise RuntimeError(\n            "No scrypt shadow entries found."\n        )\n\n    return entries[0]\n\n\n@mcp.tool()\ndef crack_shard(\n    agent_id: str,\n    shard_file: str,\n) -> dict:\n\n    entries = _parse_shadow_pdf()\n\n    if not entries:\n        raise RuntimeError(\n            "No scrypt shadow entries found."\n        )\n\n    target = entries[0]\n\n    shard_path = Path(\n        shard_file\n    )\n\n    if not shard_path.exists():\n        raise FileNotFoundError(\n            f"Missing shard: {shard_path}"\n        )\n\n    shard = [\n        line.strip()\n        for line\n        in shard_path.read_text().splitlines()\n        if line.strip()\n    ]\n\n    start = time.perf_counter()\n\n    password = None\n    attempts = 0\n\n    _write_progress(\n        agent_id,\n        attempts,\n        len(shard),\n        start,\n        "running",\n    )\n\n    # scrypt is memory-hard and intentionally expensive.\n    # Update progress frequently enough to show movement without\n    # adding material overhead.\n    REPORT_EVERY = 50\n\n    for word in shard:\n\n        attempts += 1\n\n        if _verify_candidate(\n            word,\n            target,\n        ):\n\n            password = word\n\n            _write_progress(\n                agent_id,\n                attempts,\n                len(shard),\n                start,\n                "found",\n                found=True,\n                password=password,\n            )\n\n            break\n\n        if attempts % REPORT_EVERY == 0:\n\n            _write_progress(\n                agent_id,\n                attempts,\n                len(shard),\n                start,\n                "running",\n            )\n\n    elapsed = (\n        time.perf_counter()\n        - start\n    )\n\n    if password is None:\n\n        _write_progress(\n            agent_id,\n            attempts,\n            len(shard),\n            start,\n            "finished",\n            found=False,\n        )\n\n    result = {\n        "agent_id": agent_id,\n        "user": target["user"],\n        "password": password,\n        "found": (\n            password is not None\n        ),\n        "attempts": attempts,\n        "shard_size": len(shard),\n        "elapsed_seconds": elapsed,\n        "n": target["n"],\n        "r": target["r"],\n        "p": target["p"],\n    }\n\n    result_file = (\n        RESULTS\n        / f"{agent_id}_result.json"\n    )\n\n    result_file.write_text(\n        json.dumps(\n            result,\n            indent=2,\n        )\n    )\n\n    return result\n\n\nif __name__ == "__main__":\n\n    mcp.run(\n        transport="stdio"\n    )\n'

mcp_server_path = (
    SRC
    / "mcp_server.py"
)

mcp_server_path.write_text(
    mcp_code
)

# Remove stale progress files from earlier runs.
for old_file in PROGRESS.glob("Worker-Agent-*.json"):
    old_file.unlink()

for old_file in PROGRESS.glob("Worker-Agent-*.tmp"):
    old_file.unlink()

print(
    "scrypt MCP server written:"
)

print(
    mcp_server_path
)

scrypt MCP server written:
/content/drive/MyDrive/Task2_Efficient_MultiAgent_Scrypt/src/mcp_server.py


In [ ]:
# Smoke test generated MCP server before running the agent system.

import subprocess

test = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import sys; "
            f"sys.path.insert(0, {repr(str(SRC))}); "
            "import mcp_server; "
            "print('mcp_server import OK')"
        ),
    ],
    capture_output=True,
    text=True,
)

print(test.stdout)

if test.returncode != 0:

    print(test.stderr)

    raise RuntimeError(
        "MCP server import failed."
    )

mcp_server import OK



## Load the local model

In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

tokenizer = (
    AutoTokenizer.from_pretrained(
        str(MODEL_DIR)
    )
)

model = (
    AutoModelForCausalLM.from_pretrained(
        str(MODEL_DIR),
        torch_dtype="auto",
        device_map="auto",
    )
)

print(
    "Local agent model loaded."
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Local agent model loaded.


In [ ]:
def ask_agent(
    agent_name,
    role,
    task,
):

    messages = [
        {
            "role": "system",
            "content": (
                "You are an independent agent "
                "in a controlled classroom multi-agent system. "
                "Return only valid JSON. "
                "Do not invent tool arguments."
            ),
        },
        {
            "role": "user",
            "content": f"""
Agent: {agent_name}

Role:
{role}

Task:
{task}

Return only JSON.
""",
        },
    ]

    formatted = (
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    )

    inputs = tokenizer(
        formatted,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
        )

    reply = tokenizer.decode(
        output[
            0
        ][
            inputs[
                "input_ids"
            ].shape[1]:
        ],
        skip_special_tokens=True,
    ).strip()

    print(
        f"{agent_name}:",
        reply,
    )

    match = re.search(
        r"\{.*\}",
        reply,
        re.S,
    )

    if not match:
        return None

    try:

        return json.loads(
            match.group(0)
        )

    except Exception:

        return None

## Coordinator Agent

In [ ]:
coordinator_raw = ask_agent(
    "Coordinator-Agent",
    "Coordinate four parallel scrypt workers.",
    f"""
Target only first user:
{first["User"]}

scrypt parameters:
N={first["N"]}
r={first["r"]}
p={first["p"]}

There are exactly four worker agents.

Each worker has one unique candidate shard.

Approve this parallel plan.

Return:
{{"action":"parallelize","workers":4}}
""",
)

coordinator_plan = {
    "action": "parallelize",
    "workers": NUM_WORKER_AGENTS,
}

print()
print(
    "Validated coordinator plan:"
)
print(
    coordinator_plan
)

Coordinator-Agent: ```json
{
  "action": "parallelize",
  "workers": 4
}
```

Validated coordinator plan:
{'action': 'parallelize', 'workers': 4}


## Worker Agent plans

In [ ]:
worker_plans = []

for i in range(
    NUM_WORKER_AGENTS
):

    agent_id = (
        f"Worker-Agent-{i+1}"
    )

    shard_file = str(
        shard_files[i]
    )

    raw_plan = ask_agent(
        agent_id,
        "Test one unique scrypt candidate shard.",
        f"""
Target user:
{first["User"]}

scrypt parameters:
N={first["N"]}
r={first["r"]}
p={first["p"]}

Your shard file:
{shard_file}

Available MCP tool:
crack_shard(agent_id, shard_file)

Return:
{{"tool":"crack_shard","arguments":{{"agent_id":"{agent_id}","shard_file":"{shard_file}"}}}}
""",
    )

    validated_plan = {
        "agent_id": agent_id,
        "tool": "crack_shard",
        "arguments": {
            "agent_id": agent_id,
            "shard_file": shard_file,
        },
    }

    worker_plans.append(
        validated_plan
    )

print()
print(
    "Validated worker plans:"
)

for plan in worker_plans:
    print(plan)

Worker-Agent-1: ```json
{
  "tool": "crack_shard",
  "arguments": {
    "agent_id": "Worker-Agent-1",
    "shard_file": "/content/drive/MyDrive/Task2_Efficient_MultiAgent_Scrypt/inputs/candidate_shard_0.txt"
  }
}
```
Worker-Agent-2: ```json
{
  "tool": "crack_shard",
  "arguments": {
    "agent_id": "Worker-Agent-2",
    "shard_file": "/content/drive/MyDrive/Task2_Efficient_MultiAgent_Scrypt/inputs/candidate_shard_1.txt"
  }
}
```
Worker-Agent-3: ```json
{
  "tool": "crack_shard",
  "arguments": {
    "agent_id": "Worker-Agent-3",
    "shard_file": "/content/drive/MyDrive/Task2_Efficient_MultiAgent_Scrypt/inputs/candidate_shard_2.txt"
  }
}
```
Worker-Agent-4: ```json
{
  "tool": "crack_shard",
  "arguments": {
    "agent_id": "Worker-Agent-4",
    "shard_file": "/content/drive/MyDrive/Task2_Efficient_MultiAgent_Scrypt/inputs/candidate_shard_3.txt"
  }
}
```

Validated worker plans:
{'agent_id': 'Worker-Agent-1', 'tool': 'crack_shard', 'arguments': {'agent_id': 'Worker-Agent-1', 'shar

## Run the 4 workers concurrently through MCP

In [ ]:
from mcp import (
    ClientSession,
    StdioServerParameters,
)

from mcp.client.stdio import (
    stdio_client,
)


async def run_worker(
    plan
):

    agent_id = plan[
        "agent_id"
    ]

    params = (
        StdioServerParameters(
            command=sys.executable,
            args=[
                str(
                    SRC
                    / "mcp_server.py"
                )
            ],
        )
    )

    log_path = (
        RESULTS
        / f"{agent_id}_stderr.log"
    )

    with open(
        log_path,
        "w",
    ) as errlog:

        async with stdio_client(
            params,
            errlog=errlog,
        ) as (
            read,
            write,
        ):

            async with ClientSession(
                read,
                write,
            ) as session:

                await session.initialize()

                await session.call_tool(
                    plan["tool"],
                    arguments=plan[
                        "arguments"
                    ],
                )

                result_file = (
                    RESULTS
                    / f"{agent_id}_result.json"
                )

                if result_file.exists():

                    return json.loads(
                        result_file.read_text()
                    )

                return {
                    "agent_id": agent_id,
                    "found": False,
                    "error": (
                        "Worker returned without "
                        "creating a result file."
                    ),
                }


def _fmt_seconds(
    seconds
):

    if seconds is None:
        return "--"

    seconds = max(
        0,
        int(seconds),
    )

    minutes, sec = divmod(
        seconds,
        60,
    )

    hours, minutes = divmod(
        minutes,
        60,
    )

    if hours:
        return (
            f"{hours}h "
            f"{minutes:02d}m "
            f"{sec:02d}s"
        )

    return (
        f"{minutes}m "
        f"{sec:02d}s"
    )


def print_live_progress():

    rows = []

    for i in range(
        NUM_WORKER_AGENTS
    ):

        agent_id = (
            f"Worker-Agent-{i+1}"
        )

        progress_file = (
            PROGRESS
            / f"{agent_id}.json"
        )

        if not progress_file.exists():

            rows.append(
                {
                    "Agent": agent_id,
                    "Status": "starting",
                    "Attempts": 0,
                    "Total": len(shards[i]),
                    "%": 0.0,
                    "checks/s": 0.0,
                    "ETA": "--",
                }
            )

            continue

        try:

            p = json.loads(
                progress_file.read_text()
            )

            rows.append(
                {
                    "Agent": agent_id,
                    "Status": p.get(
                        "status",
                        "?",
                    ),
                    "Attempts": p.get(
                        "attempts",
                        0,
                    ),
                    "Total": p.get(
                        "shard_size",
                        0,
                    ),
                    "%": round(
                        p.get(
                            "percent_complete",
                            0.0,
                        ),
                        2,
                    ),
                    "checks/s": round(
                        p.get(
                            "checks_per_second",
                            0.0,
                        ),
                        2,
                    ),
                    "ETA": _fmt_seconds(
                        p.get(
                            "eta_seconds"
                        )
                    ),
                }
            )

        except Exception:

            rows.append(
                {
                    "Agent": agent_id,
                    "Status": "reading",
                    "Attempts": 0,
                    "Total": len(shards[i]),
                    "%": 0.0,
                    "checks/s": 0.0,
                    "ETA": "--",
                }
            )

    from IPython.display import (
        clear_output,
        display,
    )

    clear_output(
        wait=True
    )

    print(
        "LIVE SCRYPT WORKER PROGRESS"
    )

    print(
        "Updated every 5 seconds"
    )

    print()

    display(
        pd.DataFrame(
            rows
        )
    )


async def monitor_workers(
    tasks
):

    while True:

        print_live_progress()

        if all(
            task.done()
            for task in tasks
        ):
            break

        await asyncio.sleep(
            5
        )

    print_live_progress()


parallel_start = (
    time.perf_counter()
)

tasks = [
    asyncio.create_task(
        run_worker(
            plan
        )
    )
    for plan
    in worker_plans
]

await monitor_workers(
    tasks
)

worker_results = (
    await asyncio.gather(
        *tasks
    )
)

parallel_elapsed = (
    time.perf_counter()
    - parallel_start
)

print()
print(
    "All four workers finished."
)

print(
    "Parallel wall-clock time:",
    round(
        parallel_elapsed,
        3,
    ),
    "seconds",
)

LIVE SCRYPT WORKER PROGRESS
Updated every 5 seconds



,Agent,Status,Attempts,Total,%,checks/s,ETA
0,Worker-Agent-1,finished,33787,33787,100.00,18.37,0m 00s
1,Worker-Agent-2,found,32964,33786,97.57,18.73,0m 43s
2,Worker-Agent-3,finished,33786,33786,100.00,18.58,0m 00s
3,Worker-Agent-4,finished,33786,33786,100.00,18.76,0m 00s



All four workers finished.
Parallel wall-clock time: 1843.363 seconds


## Worker results

In [ ]:
worker_df = pd.DataFrame(
    worker_results
)

display(
    worker_df
)

winners = [
    result
    for result
    in worker_results
    if result.get(
        "found"
    ) is True
]

if not winners:

    print(
        "No worker reported success."
    )

    for i in range(
        NUM_WORKER_AGENTS
    ):

        agent_id = (
            f"Worker-Agent-{i+1}"
        )

        log_path = (
            RESULTS
            / f"{agent_id}_stderr.log"
        )

        if log_path.exists():

            log_text = (
                log_path.read_text()
            )

            if log_text.strip():

                print()
                print(
                    f"--- {agent_id} stderr ---"
                )

                print(
                    log_text
                )

    raise RuntimeError(
        "No worker found the password."
    )

winner = winners[0]

print()
print(
    "Winning agent:",
    winner["agent_id"],
)

print(
    "Candidate:",
    winner["password"],
)

,agent_id,user,password,found,attempts,shard_size,elapsed_seconds,n,r,p
0,Worker-Agent-1,Bilbo,NaN,False,33787,33787,1839.045675,16384,8,1
1,Worker-Agent-2,Bilbo,welcome,True,32964,33786,1759.639567,16384,8,1
2,Worker-Agent-3,Bilbo,NaN,False,33786,33786,1817.929682,16384,8,1
3,Worker-Agent-4,Bilbo,NaN,False,33786,33786,1800.805021,16384,8,1



Winning agent: Worker-Agent-2
Candidate: welcome


## Verifier Agent

In [ ]:
import base64
import hashlib

verifier_raw = ask_agent(
    "Verifier-Agent",
    "Independently verify the winning scrypt candidate.",
    f"""
Worker:
{winner["agent_id"]}

Target user:
{first["User"]}

Verify the returned candidate against
the original scrypt hash using:
N={first["N"]}
r={first["r"]}
p={first["p"]}

Return:
{{"action":"verify"}}
""",
)

candidate = winner[
    "password"
]

expected = base64.b64decode(
    first["Hash value"]
)

derived = hashlib.scrypt(
    candidate.encode("utf-8"),
    salt=first["Salt"].encode("utf-8"),
    n=first["N"],
    r=first["r"],
    p=first["p"],
    dklen=len(expected),
    maxmem=1024 * 1024 * 1024,
)

verified = (
    derived == expected
)

if not verified:
    raise RuntimeError(
        "Verifier rejected scrypt candidate."
    )

print(
    "Verifier confirmed:",
    verified,
)

Verifier-Agent: ```json
{
  "action": "verify"
}
```
Verifier confirmed: True


## Final report

In [ ]:
final_result = {
    "user": first[
        "User"
    ],
    "password": candidate,
    "verified": verified,
    "winning_agent": winner[
        "agent_id"
    ],
    "worker_agents": (
        NUM_WORKER_AGENTS
    ),
    "total_logical_agents": (
        NUM_WORKER_AGENTS
        + 2
    ),
    "parallel_wall_clock_seconds": (
        parallel_elapsed
    ),
    "algorithm": first[
        "Algorithm"
    ],
    "N": first[
        "N"
    ],
    "r": first[
        "r"
    ],
    "p": first[
        "p"
    ],
    "salt": first[
        "Salt"
    ],
    "worker_results": (
        worker_results
    ),
}

final_path = (
    RESULTS
    / "efficient_multiagent_scrypt_first_user_result.json"
)

final_path.write_text(
    json.dumps(
        final_result,
        indent=2,
    )
)

elapsed = final_result[
    "parallel_wall_clock_seconds"
]

minutes = int(
    elapsed // 60
)

seconds = (
    elapsed % 60
)

print(
    "=========================================="
)

print(
    "EFFICIENT MULTI-AGENT SCRYPT RESULT"
)

print(
    "=========================================="
)

print(
    f"User: "
    f"{final_result['user']}"
)

print(
    f"Password: "
    f"{final_result['password']}"
)

print(
    f"Verified: "
    f"{final_result['verified']}"
)

print(
    f"Winning agent: "
    f"{final_result['winning_agent']}"
)

print(
    f"Worker agents: "
    f"{final_result['worker_agents']}"
)

print(
    f"Total logical agents: "
    f"{final_result['total_logical_agents']}"
)

print(
    "Parallel wall-clock cracking time:",
    f"{minutes} minutes {seconds:.1f} seconds",
)

print(
    f"scrypt N: "
    f"{final_result['N']}"
)

print(
    f"scrypt r: "
    f"{final_result['r']}"
)

print(
    f"scrypt p: "
    f"{final_result['p']}"
)

print()
print(
    "Saved to:"
)

print(
    final_path
)

EFFICIENT MULTI-AGENT SCRYPT RESULT
User: Bilbo
Password: welcome
Verified: True
Winning agent: Worker-Agent-2
Worker agents: 4
Total logical agents: 6
Parallel wall-clock cracking time: 30 minutes 43.4 seconds
scrypt N: 16384
scrypt r: 8
scrypt p: 1

Saved to:
/content/drive/MyDrive/Task2_Efficient_MultiAgent_Scrypt/results/efficient_multiagent_scrypt_first_user_result.json


## Output location

Successful runs save the final result to:

`MyDrive/Task2_Efficient_MultiAgent_Scrypt/results/efficient_multiagent_first_user_result.json`

The worker-specific JSON files and stderr logs are also kept in the same `results` folder.

In [2]:
# =====================================================================
# Tool tests (fully standalone) - safe to run as the LAST cell.
#
# Exercises every tool declared in mcp_server.py:
#   - get_first_user()
#   - crack_shard(agent_id, shard_file)
#
# This cell depends on NOTHING earlier in the notebook: not the mounted
# Google Drive, not the generated mcp_server.py, not any prior variable.
# If the real mcp_server.py is importable it uses that; otherwise it
# writes an identical copy to a local temp folder. In both cases the
# shadow source is replaced by a SYNTHETIC scrypt target generated here
# (low work factor N=1024), so the tests are fast, deterministic, and
# never touch Drive or the real password.
# =====================================================================

import base64
import hashlib
import importlib
import sys
import tempfile
from pathlib import Path

# --- Ensure required third-party deps are importable ----------------
for _pkg, _imp in (("mcp<2", "mcp.server.fastmcp"), ("pypdf", "pypdf")):
    try:
        importlib.import_module(_imp)
    except Exception:
        import subprocess
        subprocess.run(
            [sys.executable, "-m", "pip", "-q", "install", _pkg],
            check=True,
        )

# --- Obtain the mcp_server module (real if present, else local copy) -
_EXPECTED_SRC = Path(
    "/content/drive/MyDrive/Task2_Efficient_MultiAgent_Scrypt/src"
)
if _EXPECTED_SRC.exists() and str(_EXPECTED_SRC) not in sys.path:
    sys.path.insert(0, str(_EXPECTED_SRC))

try:
    import mcp_server
    importlib.reload(mcp_server)
    _source = "real mcp_server.py"
except ModuleNotFoundError:
    # Self-provision an identical copy in a local temp project.
    _tmp_project = Path(tempfile.mkdtemp(prefix="scrypt_tooltest_"))
    _SERVER_SRC = '\nfrom pathlib import Path\nimport base64\nimport hashlib\nimport json\nimport time\n\nfrom pypdf import PdfReader\nfrom mcp.server.fastmcp import FastMCP\n\n\nPROJECT = Path(\n    "/content/drive/MyDrive/Task2_Efficient_MultiAgent_Scrypt"\n)\n\nSHADOW_PDF = (\n    PROJECT\n    / "inputs"\n    / "shadow_scrypt.pdf"\n)\n\nRESULTS = PROJECT / "results"\nPROGRESS = PROJECT / "progress"\n\nRESULTS.mkdir(\n    parents=True,\n    exist_ok=True,\n)\n\nPROGRESS.mkdir(\n    parents=True,\n    exist_ok=True,\n)\n\nmcp = FastMCP(\n    "efficient-multiagent-scrypt"\n)\n\n\ndef _parse_shadow_pdf():\n\n    reader = PdfReader(\n        str(SHADOW_PDF)\n    )\n\n    text = "\\n".join(\n        (page.extract_text() or "")\n        for page in reader.pages\n    )\n\n    entries = []\n\n    for raw in text.splitlines():\n\n        line = raw.strip().strip(\'"\')\n\n        if ":$scrypt$" not in line:\n            continue\n\n        user, scrypt_record = line.split(\n            ":",\n            1,\n        )\n\n        parts = scrypt_record.split("$")\n\n        if len(parts) != 7:\n            continue\n\n        _, algorithm, n_text, r_text, p_text, salt, hash_b64 = parts\n\n        if algorithm != "scrypt":\n            continue\n\n        entries.append(\n            {\n                "user": user,\n                "algorithm": algorithm,\n                "n": int(n_text),\n                "r": int(r_text),\n                "p": int(p_text),\n                "salt": salt,\n                "hash_b64": hash_b64,\n            }\n        )\n\n    return entries\n\n\ndef _verify_candidate(\n    password,\n    target,\n):\n\n    expected = base64.b64decode(\n        target["hash_b64"]\n    )\n\n    derived = hashlib.scrypt(\n        password.encode("utf-8"),\n        salt=target["salt"].encode("utf-8"),\n        n=target["n"],\n        r=target["r"],\n        p=target["p"],\n        dklen=len(expected),\n        maxmem=1024 * 1024 * 1024,\n    )\n\n    return derived == expected\n\n\ndef _write_progress(\n    agent_id,\n    attempts,\n    shard_size,\n    start_time,\n    status,\n    found=False,\n    password=None,\n):\n\n    elapsed = max(\n        time.perf_counter() - start_time,\n        1e-9,\n    )\n\n    rate = attempts / elapsed\n\n    remaining = max(\n        shard_size - attempts,\n        0,\n    )\n\n    eta_seconds = (\n        remaining / rate\n        if rate > 0\n        else None\n    )\n\n    progress = {\n        "agent_id": agent_id,\n        "status": status,\n        "attempts": attempts,\n        "shard_size": shard_size,\n        "percent_complete": (\n            100.0 * attempts / shard_size\n            if shard_size\n            else 100.0\n        ),\n        "elapsed_seconds": elapsed,\n        "checks_per_second": rate,\n        "eta_seconds": eta_seconds,\n        "found": found,\n        "password": password,\n    }\n\n    tmp = PROGRESS / f"{agent_id}.tmp"\n    final = PROGRESS / f"{agent_id}.json"\n\n    tmp.write_text(\n        json.dumps(\n            progress,\n            indent=2,\n        )\n    )\n\n    tmp.replace(\n        final\n    )\n\n\n@mcp.tool()\ndef get_first_user() -> dict:\n\n    entries = _parse_shadow_pdf()\n\n    if not entries:\n        raise RuntimeError(\n            "No scrypt shadow entries found."\n        )\n\n    return entries[0]\n\n\n@mcp.tool()\ndef crack_shard(\n    agent_id: str,\n    shard_file: str,\n) -> dict:\n\n    entries = _parse_shadow_pdf()\n\n    if not entries:\n        raise RuntimeError(\n            "No scrypt shadow entries found."\n        )\n\n    target = entries[0]\n\n    shard_path = Path(\n        shard_file\n    )\n\n    if not shard_path.exists():\n        raise FileNotFoundError(\n            f"Missing shard: {shard_path}"\n        )\n\n    shard = [\n        line.strip()\n        for line\n        in shard_path.read_text().splitlines()\n        if line.strip()\n    ]\n\n    start = time.perf_counter()\n\n    password = None\n    attempts = 0\n\n    _write_progress(\n        agent_id,\n        attempts,\n        len(shard),\n        start,\n        "running",\n    )\n\n    # scrypt is memory-hard and intentionally expensive.\n    # Update progress frequently enough to show movement without\n    # adding material overhead.\n    REPORT_EVERY = 50\n\n    for word in shard:\n\n        attempts += 1\n\n        if _verify_candidate(\n            word,\n            target,\n        ):\n\n            password = word\n\n            _write_progress(\n                agent_id,\n                attempts,\n                len(shard),\n                start,\n                "found",\n                found=True,\n                password=password,\n            )\n\n            break\n\n        if attempts % REPORT_EVERY == 0:\n\n            _write_progress(\n                agent_id,\n                attempts,\n                len(shard),\n                start,\n                "running",\n            )\n\n    elapsed = (\n        time.perf_counter()\n        - start\n    )\n\n    if password is None:\n\n        _write_progress(\n            agent_id,\n            attempts,\n            len(shard),\n            start,\n            "finished",\n            found=False,\n        )\n\n    result = {\n        "agent_id": agent_id,\n        "user": target["user"],\n        "password": password,\n        "found": (\n            password is not None\n        ),\n        "attempts": attempts,\n        "shard_size": len(shard),\n        "elapsed_seconds": elapsed,\n        "n": target["n"],\n        "r": target["r"],\n        "p": target["p"],\n    }\n\n    result_file = (\n        RESULTS\n        / f"{agent_id}_result.json"\n    )\n\n    result_file.write_text(\n        json.dumps(\n            result,\n            indent=2,\n        )\n    )\n\n    return result\n\n\nif __name__ == "__main__":\n\n    mcp.run(\n        transport="stdio"\n    )\n'
    _SERVER_SRC = _SERVER_SRC.replace(
        "/content/drive/MyDrive/Task2_Efficient_MultiAgent_Scrypt",
        str(_tmp_project),
    )
    _tmp_src = _tmp_project / "src"
    _tmp_src.mkdir(parents=True, exist_ok=True)
    (_tmp_src / "mcp_server.py").write_text(_SERVER_SRC)
    sys.path.insert(0, str(_tmp_src))
    import mcp_server
    importlib.reload(mcp_server)
    _source = f"local copy at {_tmp_src / 'mcp_server.py'}"

print(f"Testing: {_source}\n")

# Work inside the module's own project folders (temp or Drive).
_RESULTS = mcp_server.RESULTS
_PROGRESS = mcp_server.PROGRESS
_TEST_DIR = Path(tempfile.mkdtemp(prefix="scrypt_shards_"))

_passed = []
_failed = []


def _check(name, condition, detail=""):
    if condition:
        _passed.append(name)
        print(f"PASS  {name}")
    else:
        _failed.append((name, detail))
        print(f"FAIL  {name}  ->  {detail}")


# --- Synthetic target shared by BOTH tools (no PDF / Drive needed) --
_TEST_PW = "secretword"
_TEST_SALT = "TestSalt0123456789ab"
_TEST_N, _TEST_R, _TEST_P = 1024, 8, 1

_expected = hashlib.scrypt(
    _TEST_PW.encode("utf-8"),
    salt=_TEST_SALT.encode("utf-8"),
    n=_TEST_N,
    r=_TEST_R,
    p=_TEST_P,
    dklen=32,
    maxmem=1024 * 1024 * 1024,
)

_synthetic_target = {
    "user": "TestUser",
    "algorithm": "scrypt",
    "n": _TEST_N,
    "r": _TEST_R,
    "p": _TEST_P,
    "salt": _TEST_SALT,
    "hash_b64": base64.b64encode(_expected).decode("ascii"),
}

_real_parse = mcp_server._parse_shadow_pdf
mcp_server._parse_shadow_pdf = lambda: [_synthetic_target]

try:
    # =================================================================
    # Tool 1: get_first_user()
    # =================================================================
    _first = mcp_server.get_first_user()

    _check(
        "get_first_user: returns a dict",
        isinstance(_first, dict),
        f"got {type(_first).__name__}",
    )
    _check(
        "get_first_user: has all expected keys",
        set(_first) >= {"user", "algorithm", "n", "r", "p", "salt", "hash_b64"},
        f"keys={sorted(_first)}",
    )
    _check(
        "get_first_user: algorithm == 'scrypt'",
        _first.get("algorithm") == "scrypt",
        f"algorithm={_first.get('algorithm')!r}",
    )
    _check(
        "get_first_user: N, r, p are ints",
        all(isinstance(_first.get(k), int) for k in ("n", "r", "p")),
        f"n={_first.get('n')!r} r={_first.get('r')!r} p={_first.get('p')!r}",
    )
    _check(
        "get_first_user: returns the first (only) shadow entry",
        _first == _synthetic_target,
        "returned entry differs from entries[0]",
    )

    # =================================================================
    # Tool 2: crack_shard(agent_id, shard_file)
    # =================================================================
    # --- positive: target password present in the shard ---
    _found_shard = _TEST_DIR / "shard_found.txt"
    _found_shard.write_text(
        "\n".join(["alpha", "bravo", _TEST_PW, "charlie"])
    )

    _r_found = mcp_server.crack_shard(
        "Test-CrackShard-Found", str(_found_shard)
    )

    _check(
        "crack_shard: returns a dict",
        isinstance(_r_found, dict),
        f"got {type(_r_found).__name__}",
    )
    _check(
        "crack_shard: finds the target password",
        _r_found.get("found") is True and _r_found.get("password") == _TEST_PW,
        f"found={_r_found.get('found')} password={_r_found.get('password')!r}",
    )
    _check(
        "crack_shard: stops at the matching candidate (attempts == 3)",
        _r_found.get("attempts") == 3,
        f"attempts={_r_found.get('attempts')}",
    )
    _check(
        "crack_shard: echoes target user and scrypt params",
        (
            _r_found.get("user") == "TestUser"
            and _r_found.get("n") == _TEST_N
            and _r_found.get("r") == _TEST_R
            and _r_found.get("p") == _TEST_P
        ),
        f"user={_r_found.get('user')!r} "
        f"n/r/p={_r_found.get('n')}/{_r_found.get('r')}/{_r_found.get('p')}",
    )
    _check(
        "crack_shard: writes a result JSON file",
        (_RESULTS / "Test-CrackShard-Found_result.json").exists(),
        "result file not written",
    )

    # --- negative: target password absent from the shard ---
    _absent_shard = _TEST_DIR / "shard_absent.txt"
    _absent_shard.write_text("\n".join(["delta", "echo", "foxtrot"]))

    _r_absent = mcp_server.crack_shard(
        "Test-CrackShard-NotFound", str(_absent_shard)
    )

    _check(
        "crack_shard: reports not found when absent",
        _r_absent.get("found") is False and _r_absent.get("password") is None,
        f"found={_r_absent.get('found')} password={_r_absent.get('password')!r}",
    )
    _check(
        "crack_shard: scans the whole shard when not found",
        _r_absent.get("attempts") == _r_absent.get("shard_size") == 3,
        f"attempts={_r_absent.get('attempts')} "
        f"shard_size={_r_absent.get('shard_size')}",
    )

    # --- error path: missing shard file ---
    _raised = False
    try:
        mcp_server.crack_shard(
            "Test-CrackShard-Missing",
            str(_TEST_DIR / "does_not_exist.txt"),
        )
    except FileNotFoundError:
        _raised = True

    _check(
        "crack_shard: raises FileNotFoundError for a missing shard",
        _raised,
        "no FileNotFoundError raised",
    )

finally:
    # Restore the real parser and remove test artifacts.
    mcp_server._parse_shadow_pdf = _real_parse

    import shutil as _shutil

    for _p in (
        _RESULTS / "Test-CrackShard-Found_result.json",
        _RESULTS / "Test-CrackShard-NotFound_result.json",
        _PROGRESS / "Test-CrackShard-Found.json",
        _PROGRESS / "Test-CrackShard-NotFound.json",
        _PROGRESS / "Test-CrackShard-Found.tmp",
        _PROGRESS / "Test-CrackShard-NotFound.tmp",
    ):
        if _p.exists():
            _p.unlink()

    _shutil.rmtree(_TEST_DIR, ignore_errors=True)


# =====================================================================
# Summary
# =====================================================================
print()
print("=" * 50)
print(f"TOOL TESTS: {len(_passed)} passed, {len(_failed)} failed")
print("=" * 50)

if _failed:
    for _name, _detail in _failed:
        print(f"  FAILED: {_name}  ->  {_detail}")
    raise AssertionError(f"{len(_failed)} tool test(s) failed.")

print("All declared MCP tools passed their tests.")

Testing: local copy at /tmp/scrypt_tooltest__d4nz_qy/src/mcp_server.py

PASS  get_first_user: returns a dict
PASS  get_first_user: has all expected keys
PASS  get_first_user: algorithm == 'scrypt'
PASS  get_first_user: N, r, p are ints
PASS  get_first_user: returns the first (only) shadow entry
PASS  crack_shard: returns a dict
PASS  crack_shard: finds the target password
PASS  crack_shard: stops at the matching candidate (attempts == 3)
PASS  crack_shard: echoes target user and scrypt params
PASS  crack_shard: writes a result JSON file
PASS  crack_shard: reports not found when absent
PASS  crack_shard: scans the whole shard when not found
PASS  crack_shard: raises FileNotFoundError for a missing shard

TOOL TESTS: 13 passed, 0 failed
All declared MCP tools passed their tests.
